## Running this notebook on Google Colab instead of Kaggle

This is the same notebook and the same training lineage; a platform-detection cell right after the
parameters below picks Kaggle or Colab paths automatically, so nothing else in the notebook changes.
Use this when a Kaggle session's 12 h clock runs out and Colab time (A100 or L4) is available for the
rest of a stage.

**Open it:** `https://colab.research.google.com/github/0XSreekar/True-Watch-AI/blob/fix/phase1-2-complete/training/notebooks/kaggle_train.ipynb`

**1. Pick the GPU.** Runtime -> Change runtime type -> GPU -> A100 (Colab Pro/Pro+) or L4. Either
works; `DEVICE = "0"` uses whichever single GPU Colab attaches.

**2. Add secrets.** Left sidebar, key icon -> add, with "Notebook access" switched on for each:
  - `KAGGLE_API_TOKEN` (required): the contents of the `access_token` your Kaggle account page's
    API section gives you, or set `KAGGLE_USERNAME` + `KAGGLE_KEY` instead (the values from
    `kaggle.json`). Used to download the dataset shards and this notebook's earlier Kaggle output.
  - `HF_TOKEN` and `HF_MODEL_REPO` (optional): only needed to push the exported ONNX to the
    Hugging Face Hub from Colab instead of from the Mac afterwards.

**3. Run all.** Runtime -> Run all. The platform cell mounts Google Drive at `/content/drive` (a
one-time consent popup) and everything downloaded or restored from Drive lands under
`/content/drive/MyDrive/truewatch`, so it survives a Colab disconnect. `KAGGLE_BUILD_KERNEL` and
`KAGGLE_PREV_KERNELS` below name the Kaggle kernels this session's inputs come from; change them if
the build or the earlier training session used a different kernel slug.

**4. If Colab disconnects.** Reopen the notebook and Run all again. The platform cell restores
`runs/` from Drive before training resumes there, and the dataset-fetch cell only asks Kaggle for
shard tars that are missing or the wrong size, so neither step repeats work already done.

**Session length.** `COLAB_SESSION_HOURS` in the platform cell defaults to 12 h (a Colab Pro
session); raise it for Pro+'s longer sessions or lower it for the free tier's shorter, unpredictable
ones -- the same budget math the Kaggle path uses (`remaining_train_hours()`, `--max-hours`) then
paces the stage to whatever is actually available.


# TRUEWATCH: YOLO11-s fine-tune on Kaggle (Phase 2)

One detector, one head, mixed day and IR data, trained in two stages of one lineage.
Stage 1 (`yolo11s_day.yaml`) is the bulk of training on the full corpus. Stage 2 (`yolo11s_ir.yaml`) starts from
stage 1's best weights, repeats LWIR images twice per epoch and lowers the learning rate. It is not a separate IR model.
The reasoning is in `training/README.md` and at the top of `training/configs/yolo11s_day.yaml`.

**Before you run it**

1. Settings: Accelerator = GPU T4 x2 or P100 (one GPU is used, see `DEVICE`). Internet = On (the code is cloned from GitHub,
   the COCO weights are downloaded and the AMP check fetches a tiny model).
2. Add data: the output of `datasets/notebooks/kaggle_build.ipynb`, i.e. the `truewatch_ds/` folder (YOLO tree, `data.yaml`,
   `manifests/`, `reports/`). It is found wherever Kaggle mounts it, by searching `/kaggle/input/**/truewatch_ds/data.yaml`.
3. From the second session on, also add this notebook's previous output. The run directories and the hard-set baseline are
   restored from it and training resumes from the last good checkpoint.
4. Optional secrets (Add-ons, Secrets): `HF_TOKEN` (and `HF_MODEL_REPO`, or set the parameter below) uploads the exported ONNX to
   the Hugging Face Hub; without it nothing is uploaded and the upload can be done from the Mac. `KAGGLE_USERNAME` and `KAGGLE_KEY`
   publish the results dataset.

**Time.** One 12 h clock is shared by everything in the session. `NB_START` is taken in the first cell; each training stage is
given only the time that is left, minus `RESERVE_HOURS` (at least 45 min kept for evaluation, export and packaging), and
`train.py` stops at the epoch boundary where one more epoch would cross its deadline. Stage 2 is not started unless one estimated
epoch still fits. Evaluation, export and upload run only when the final stage is complete, never on a half-trained model.

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

NB_START = time.time()             # the session clock every budget below is measured against

GIT_URL = "https://github.com/0XSreekar/True-Watch-AI.git"
GIT_REF = "fix/phase1-2-complete"  # branch or tag that holds training/
INPUT_ROOT = Path("/kaggle/input")   # where Kaggle mounts attached datasets and earlier outputs
WORK = Path("/kaggle/working")
CODE = WORK / "truewatch"
RUNS = WORK / "runs"               # run directories; kept by Save Version and re-attached next session
OUT = WORK / "outputs"             # packaged as the results dataset at the end
DEVICE = "0"                       # ONE GPU. On "T4 x2" the second T4 stays idle: train.py refuses "0,1" because Ultralytics runs
                                   # multi-GPU training in child processes built from a generated script, which carry neither
                                   # train.py's callbacks (freeze schedule, last_good.pt, train_log.csv, the deadline) nor its trainer
                                   # (the infrared-only augmentation hook). Single-GPU is the verified path.
SESSION_HOURS = 12.0               # Kaggle's hard cap per session
RESERVE_HOURS = 0.75               # kept free after training: evaluate, hard-set gate, sweep, export, metrics, packaging (>= 45 min)
TRAIN_MARGIN_HOURS = 0.25          # per train.py call: start-up (label scan and cache) and Ultralytics' final validation of best.pt
EPOCH_SAFETY = 1.10                # an epoch is assumed to take 10% longer than the slowest recent one
IR_EPOCH_FACTOR = 1.5              # stage 2 epoch / stage 1 epoch before stage 2 has its own log (LWIR repeated: ~1.4x the list)
RUN_SMOKE_TEST = "auto"            # "auto": only in a session with no earlier run attached; True / False to force
RUN_STAGE_2 = True                 # when True, evaluation/export/upload wait for stage 2 to be complete
HF_MODEL_REPO = ""                 # e.g. "<hf-user>/truewatch-yolo11s"; when empty the HF_MODEL_REPO secret is used
UPLOAD_IF_GATE_FAILS = False       # a failed hard-set regression gate blocks the Hugging Face upload unless this is True

# A LINEAGE is one training history. "" is the original run (run directories under runs/). A named lineage such as "v2" is a
# new fine-tune on an expanded dataset: its run directories live under runs_<name>/, earlier sessions are restored only from
# runs_<name>/ (so an attached output of the original run is never mistaken for this one), and stage 1 of a fresh lineage starts
# from INIT_WEIGHTS instead of the COCO model.
LINEAGE = ""
INIT_WEIGHTS = ""                  # fresh named lineage only: a path, or a glob searched under INPUT_ROOT (e.g. "runs/ir/weights/best.pt")
STAGE1_EPOCHS = None               # fresh named lineage only: overrides the stage-1 config's epochs

_last_bar = 0.0


def elapsed_h():
    return (time.time() - NB_START) / 3600.0


def remaining_train_hours():
    """Hours a train.py call may use: what is left of the session minus the post-training reserve and the per-call margin."""
    return SESSION_HOURS - elapsed_h() - RESERVE_HOURS - TRAIN_MARGIN_HOURS


def sh(cmd, check=True, ok=(0,)):
    """Run a shell command from the repo, streaming output (progress bars throttled to one a minute). Raises on an exit not in `ok`."""
    global _last_bar
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, cwd=str(CODE) if CODE.exists() else None,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        is_bar = ("it/s]" in line) or ("s/it]" in line)
        if is_bar and time.time() - _last_bar < 60:
            continue
        if is_bar:
            _last_bar = time.time()
        print(line, end="", flush=True)
    code = proc.wait()
    if check and code not in ok:
        raise RuntimeError(f"command failed with exit code {code}: {cmd}")
    return code


def find_inputs(relative, max_depth=6):
    """Paths under /kaggle/input ending in `relative`, at any mount depth up to max_depth (Kaggle's mount layout varies)."""
    hits = []
    for depth in range(1, max_depth + 1):
        hits += glob.glob(str(Path(INPUT_ROOT, *(["*"] * depth), relative)))
    return sorted(set(hits))


def earlier_run_logs(stage):
    """train_log.csv files of an earlier session's run for `stage` (never the smoke test's or the dry run's scratch dirs)."""
    return [Path(p) for p in find_inputs(f"{RUNS.name}/{stage}/train_log.csv") if "/smoke/" not in p and "/dryrun/" not in p]


def read_state(stage):
    path = RUNS / stage / "run_state.json"
    return json.loads(path.read_text()) if path.exists() else {}


def epoch_hours(stage):
    """Slowest of the last three logged epochs of a stage, in hours, or None before its first epoch."""
    path = RUNS / stage / "train_log.csv"
    if not path.exists():
        return None
    import csv
    with path.open(newline="") as fh:
        values = [float(r["epoch_s"]) for r in csv.DictReader(fh) if (r.get("epoch_s") or "").strip()]
    return max(values[-3:]) / 3600.0 if values else None


def secret(name):
    """A Kaggle secret (Add-ons, Secrets), else an environment variable, else ''."""
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name, "")


def clock(label):
    print(f"[clock] {label}: {elapsed_h():.2f} h used of {SESSION_HOURS:g} h; {remaining_train_hours():.2f} h available for training")

In [ ]:
# --- Platform: Kaggle or Colab -----------------------------------------------------------------
# Kaggle mounts attached data at /kaggle/input and sets KAGGLE_KERNEL_RUN_TYPE; Colab has no such
# mount and instead exposes `google.colab`. Everything below this cell (find_inputs, unpack_shards,
# restore, the train.py calls, evaluate/export/package) is unchanged either way: it only ever reads
# INPUT_ROOT, WORK, CODE, RUNS, OUT and secret(), which this cell points at the right places.
ON_KAGGLE = Path("/kaggle/input").exists() or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if ON_KAGGLE and ON_COLAB:
    raise RuntimeError("both /kaggle/input and google.colab are present; cannot tell which platform this is")

if ON_COLAB:
    # Colab Pro sessions run ~12 h; Pro+ longer, the free tier shorter and unpredictable. Edit to match.
    COLAB_SESSION_HOURS = 12.0
    SESSION_HOURS = COLAB_SESSION_HOURS

    # The Kaggle kernels this session pulls its inputs from (see the fetch cell below).
    KAGGLE_BUILD_KERNEL = "sreekar1206/truewatch-phase1-build"     # datasets/notebooks/kaggle_build.ipynb's output
    KAGGLE_PREV_KERNELS = ["sreekar1206/truewatch-train-s1"]       # this notebook's own earlier Kaggle session(s)

    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = Path("/content/drive/MyDrive/truewatch")   # survives a Colab disconnect; RUNS below does not
    PERSIST.mkdir(parents=True, exist_ok=True)

    INPUT_ROOT = Path("/content/input")   # populated by the fetch cell below, from Kaggle kernel outputs
    WORK = Path("/content/work")
    CODE = WORK / "truewatch"
    RUNS = WORK / "runs"          # local disk, not Drive: train.py writes last.pt/last_good.pt/train_log.csv every
                                   # epoch by renaming a temp file into place, and Google Drive's FUSE mount does not
                                   # reliably make that rename atomic. RUNS is synced to PERSIST/runs after every
                                   # stage call and at packaging instead, and restored from there below.
    OUT = WORK / "outputs"
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)
    WORK.mkdir(parents=True, exist_ok=True)

    def sync_runs_to_drive():
        """Copy RUNS to Drive so a disconnect does not lose checkpoints or logs."""
        dest = PERSIST / "runs"
        dest.mkdir(parents=True, exist_ok=True)
        sh(f"rsync -a --exclude '*.tmp' {RUNS}/ {dest}/", check=False)

    def restore_runs_from_drive():
        """The reverse of sync_runs_to_drive(), run once at the start of a (re)connected session."""
        src = PERSIST / "runs"
        if src.exists() and any(src.iterdir()):
            RUNS.mkdir(parents=True, exist_ok=True)
            sh(f"rsync -a {src}/ {RUNS}/", check=False)
            print(f"restored {src} -> {RUNS}")
        else:
            print("no earlier Drive-backed run to restore")

    restore_runs_from_drive()

    def secret(name):
        """A Colab secret (key icon, left sidebar, with notebook access on), else an environment variable, else ''."""
        try:
            from google.colab import userdata
            value = userdata.get(name)
            if value:
                return value
        except Exception:      # SecretNotFoundError / NotebookAccessError, or the secrets panel is unavailable
            pass
        return os.environ.get(name, "")

    WORKERS = os.cpu_count() or None   # A100 runtimes have ~12 vCPUs; passed to train.py below when set
else:
    PERSIST = None
    KAGGLE_BUILD_KERNEL = KAGGLE_PREV_KERNELS = None

    def sync_runs_to_drive():
        pass

    def restore_runs_from_drive():
        pass

    WORKERS = None   # leave train.py's own config default on Kaggle

WORKERS_FLAG = f"--workers {WORKERS}" if WORKERS else ""
clock(f"platform detected: {'Colab' if ON_COLAB else 'Kaggle'}")

if LINEAGE:
    RUNS = RUNS.parent / f"runs_{LINEAGE}"
print(f"lineage: {LINEAGE or 'original'}; run directories under {RUNS}")


In [ ]:
sh("nvidia-smi --query-gpu=name,memory.total --format=csv; python --version; df -h /kaggle/working | tail -1", check=False)

In [ ]:
if not CODE.exists():
    code_rc = sh(f"git clone --depth 1 --branch {GIT_REF} {GIT_URL} {CODE}", check=False)
    if code_rc != 0:
        # No internet: fall back to an attached dataset that carries the repo (needs training/ and datasets/config/).
        local = find_inputs("training/scripts/train.py")
        if not local:
            raise RuntimeError("could not clone the repository and no attached dataset contains training/scripts/train.py")
        shutil.copytree(Path(local[0]).parents[2], CODE)
sh("git log -1 --oneline 2>/dev/null || echo '(code copied from an attached dataset)'", check=False)
sh("pip install -q -r training/requirements.txt")
sh("python -c \"import ultralytics, torch, albumentations as A; print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, "
   "'| cuda', torch.cuda.is_available(), '| albumentations', A.__version__)\"")
RESULTS = CODE / "training" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
clock("code ready")

### Colab only: fetch the dataset and earlier Kaggle outputs

Kaggle mounts attached datasets at `/kaggle/input` automatically; Colab has nothing attached, so the
same two Kaggle kernel outputs are downloaded here with the Kaggle CLI, into `INPUT_ROOT`. After this
cell, `find_inputs()` / `unpack_shards()` / `restore()` below see the same `truewatch_ds_shards/` and
`runs/{day,ir}` layout they see on Kaggle and need no further changes.


In [ ]:
if ON_COLAB:
    sh("pip install -q 'kaggle>=1.8'")

    kaggle_token = secret("KAGGLE_API_TOKEN")
    if kaggle_token:
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        token_path = kaggle_dir / "access_token"
        token_path.write_text(kaggle_token)
        token_path.chmod(0o600)
    else:
        kaggle_user, kaggle_key = secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY")
        if not (kaggle_user and kaggle_key):
            raise RuntimeError(
                "add a KAGGLE_API_TOKEN Colab secret (Add-ons -> key icon), or KAGGLE_USERNAME + KAGGLE_KEY, "
                "to download the dataset and earlier runs from Kaggle"
            )
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = kaggle_user, kaggle_key

    sys.path.insert(0, str(CODE / "training" / "scripts"))
    import fetch_kaggle_outputs as FK

    clock("before fetching Kaggle outputs")
    FK.fetch_dataset_shards(KAGGLE_BUILD_KERNEL, INPUT_ROOT / "build")
    for kernel in KAGGLE_PREV_KERNELS:
        FK.fetch_prev_run_outputs(kernel, INPUT_ROOT / kernel.split("/")[-1])
    clock("after fetching Kaggle outputs")
else:
    print("fetch skipped: inputs are already attached and mounted at /kaggle/input")


## 1. Find the dataset

The dataset notebook writes `truewatch_ds/` with a `data.yaml` whose paths are relative, so the folder can be mounted anywhere.
Every script below gets it as `--data-root`. If the cart gate withdrew class 4, `data.yaml` lists four names (or keeps five with
no cart label); `train.py` keeps the 5-output head either way and reports the withdrawal.

In [ ]:
import yaml

# The build notebook saves the dataset as tar shards (Kaggle drops ~170k loose output files), so an attached
# build output is verified against SHARDS.json and unpacked onto local disk first. A loose truewatch_ds/ input
# (e.g. a Kaggle Dataset made from the tree) is still used directly.
import hashlib, tarfile

LOCAL_DS_PARENT = Path("/tmp/truewatch_local")


def unpack_shards():
    index_files = find_inputs("truewatch_ds_shards/SHARDS.json")
    if not index_files:
        return None
    if len(index_files) > 1:
        raise RuntimeError(f"several dataset shard sets are attached, remove all but one: {index_files}")
    index_path = Path(index_files[0])
    index = json.loads(index_path.read_text())
    target = LOCAL_DS_PARENT / "truewatch_ds"
    marker = LOCAL_DS_PARENT / ".unpacked"
    if marker.exists() and marker.read_text() == index_path.read_text():
        return target
    if LOCAL_DS_PARENT.exists():
        shutil.rmtree(LOCAL_DS_PARENT)
    LOCAL_DS_PARENT.mkdir(parents=True)
    started = time.time()
    for shard in index["shards"]:
        path = index_path.parent / shard["name"]
        if path.stat().st_size != shard["bytes"]:
            raise RuntimeError(f"{path.name}: size {path.stat().st_size} != {shard['bytes']} recorded in SHARDS.json")
        digest = hashlib.sha256()
        with open(path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 22), b""):
                digest.update(chunk)
        if digest.hexdigest() != shard["sha256"]:
            raise RuntimeError(f"{path.name}: sha256 mismatch against SHARDS.json")
        with tarfile.open(path) as tar:
            tar.extractall(LOCAL_DS_PARENT, filter="data")
        if ON_COLAB:   # keep disk free: SHARDS.json (kept) is all the marker check below needs
            path.unlink(missing_ok=True)
    files = sum(1 for p in target.rglob("*") if p.is_file())
    if files != index["files"]:
        raise RuntimeError(f"unpacked {files} files, SHARDS.json records {index['files']}")
    marker.write_text(index_path.read_text())
    print(f"unpacked {len(index['shards'])} shards ({files} files) to {target} in {time.time() - started:.0f} s")
    return target


unpacked = unpack_shards()
candidates = [Path(p).parent for p in find_inputs("truewatch_ds/data.yaml")]
if unpacked is not None:
    candidates = [unpacked] + [c for c in candidates if c != unpacked]
candidates = [c for c in candidates if (c / "images" / "val").is_dir() and (c / "labels" / "val").is_dir()]
if not candidates:
    raise RuntimeError("no attached input holds truewatch_ds/data.yaml with images/val and labels/val. Add the output of "
                       "datasets/notebooks/kaggle_build.ipynb as an input.")
if unpacked is None and len(candidates) > 1:
    raise RuntimeError(f"several truewatch_ds inputs are attached, remove all but one: {[str(c) for c in candidates]}")
DATA_ROOT = candidates[0]  # the unpacked shards win over any loose copy
os.environ["TRUEWATCH_DATA_ROOT"] = str(DATA_ROOT)
data_yaml = yaml.safe_load((DATA_ROOT / "data.yaml").read_text())
names = data_yaml["names"]
names = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
print(f"dataset: {DATA_ROOT}\nclasses: {names}" + ("   (cart withdrawn by the cart gate)" if "cart" not in names else ""))

hard = [DATA_ROOT / "manifests" / "hard_set.txt"] + [Path(p) for p in find_inputs("truewatch_ds/manifests/hard_set.txt")]
HARD_SET = next((p for p in hard if p.is_file()), None)
index = [DATA_ROOT / "index" / "split.jsonl", DATA_ROOT / "manifests" / "split.jsonl"] + [Path(p) for p in find_inputs("truewatch_ds/index/split.jsonl")]
INDEX = next((p for p in index if p.is_file()), None)
INDEX_FLAG = f"--index {INDEX}" if INDEX else ""
print(f"hard set manifest: {HARD_SET or 'NOT FOUND (the regression gate will not run)'}")
print(f"split index: {INDEX or 'not found (lighting sub-slices and bootstrap clusters are coarser without it)'}")

EARLIER_RUN = bool(earlier_run_logs("day"))
print("an earlier session's output is attached" if EARLIER_RUN else "no earlier session attached: first session")

## 2. Smoke test (minutes, not hours)

Builds a tiny synthetic dataset and runs the real scripts end to end: train, kill and resume, evaluate, sweep, export with the
torch-versus-onnxruntime parity check, and the CPU benchmark. In `"auto"` mode it runs only in the first session.

In [ ]:
if RUN_SMOKE_TEST is True or (RUN_SMOKE_TEST == "auto" and not EARLIER_RUN):
    sh(f"python training/scripts/smoke_test.py --device {DEVICE} --workdir {WORK / 'smoke'}")
else:
    print("smoke test skipped")
clock("after smoke test")

## 3. Dataset preflight and the training plan

`--dry-run` checks the dataset (class histogram, label ids, the cart gate, duplicate filenames that would pair an image with the
wrong labels), checks that every key of `augment.yaml` will be applied, and prints the plan without training. It uses a scratch
run directory so it never collides with a restored run. A failure here raises before any GPU time is spent.

In [ ]:
sh(f"python training/scripts/train.py --stage day --data-root {DATA_ROOT} --run-dir {WORK / 'dryrun' / 'day'} --device {DEVICE} --dry-run")

## 4. Restore the previous session

Copies the most advanced earlier run for each stage, and the hard-set baseline, from an attached output. Does nothing in the first session.

In [ ]:
def restore(stage):
    dest = RUNS / stage
    if (dest / "weights").exists():
        print(f"{stage}: run directory already present at {dest}")
        return
    logs = earlier_run_logs(stage)
    if not logs:
        print(f"{stage}: no earlier session attached, starting fresh")
        return
    best = max(logs, key=lambda p: sum(1 for _ in p.open())).parent
    shutil.copytree(best, dest, ignore=shutil.ignore_patterns("*.tmp"))
    print(f"{stage}: restored {best} -> {dest}")


RUNS.mkdir(parents=True, exist_ok=True)
for stage in ("day", "ir"):
    restore(stage)

# A baseline belongs to one hard set. It is restored only when its source record names the hard set in use (by sha256); a
# baseline recorded before that field existed is trusted only in the original lineage, whose hard set it was measured on.
HARD_SET_SHA = __import__("hashlib").sha256(HARD_SET.read_bytes()).hexdigest() if HARD_SET else None
baseline_target, source_target = RESULTS / "hardset_baseline.json", RESULTS / "hardset_baseline_source.json"
if not baseline_target.exists():
    for src_json in [Path(p) for p in find_inputs("results/hardset_baseline_source.json")]:
        recorded = json.loads(src_json.read_text()).get("hard_set_sha256")
        matches = recorded == HARD_SET_SHA if recorded else not LINEAGE
        if matches and (src_json.parent / "hardset_baseline.json").exists():
            shutil.copy2(src_json.parent / "hardset_baseline.json", baseline_target)
            shutil.copy2(src_json, source_target)
            print(f"restored hard-set baseline from {src_json.parent}")
            break
        print(f"not restoring the baseline in {src_json.parent}: it was measured on a different hard set")

## 5. Stage 1: day (the bulk of training, COCO weights to the mixed corpus)

`--auto-resume` continues from the newest valid checkpoint if there is one. `--max-hours` is the time left in THIS session, so a
stage started late gets less. A non-zero exit raises; a clean stop for time is exit 0 and the next session resumes.

In [ ]:
clock("before stage 1")
state1 = read_state("day")
if state1.get("complete"):
    print("stage 1 is already complete")
else:
    budget = remaining_train_hours()
    est = epoch_hours("day")
    if est is not None and est * EPOCH_SAFETY > budget:
        print(f"stage 1 not started: one epoch takes ~{est:.2f} h and only {budget:.2f} h are available. Save Version and resume.")
    elif budget <= 0:
        print("stage 1 not started: no training time left in this session")
    else:
        init = ""
        if LINEAGE and not (RUNS / "day" / "weights").exists():
            # A fresh named lineage: stage 1 starts from INIT_WEIGHTS, not from COCO.
            if not INIT_WEIGHTS:
                raise RuntimeError(f"lineage {LINEAGE!r} has no earlier run and INIT_WEIGHTS is empty")
            hits = [Path(INIT_WEIGHTS)] if Path(INIT_WEIGHTS).is_file() else [Path(p) for p in find_inputs(INIT_WEIGHTS)]
            if len(hits) != 1:
                raise RuntimeError(f"INIT_WEIGHTS {INIT_WEIGHTS!r} matched {len(hits)} files, need exactly one: {hits}")
            init = f"--model {hits[0]}" + (f" --epochs {STAGE1_EPOCHS}" if STAGE1_EPOCHS else "")
            print(f"lineage {LINEAGE}: stage 1 starts from {hits[0]}")
        sh(f"python training/scripts/train.py --stage day --data-root {DATA_ROOT} --run-dir {RUNS / 'day'} --device {DEVICE} "
           f"--auto-resume --max-hours {budget:.3f} {WORKERS_FLAG} {init}")
        sync_runs_to_drive()
    state1 = read_state("day")
STAGE1_DONE = bool(state1.get("complete"))
print(json.dumps({k: state1.get(k) for k in ("status", "epochs_done", "epochs_total", "early_stopped", "withdrawn_classes", "note")}, indent=2))
if not STAGE1_DONE:
    print("Stage 1 is not finished. Save Version, attach this notebook's output to the next session, and run again: it resumes.")
clock("after stage 1")

## 6. Stage 2: IR emphasis (weights from stage 1, LWIR repeated, lower learning rate)

Started only when stage 1 is complete and at least one estimated stage-2 epoch fits in what is left of the session.

In [ ]:
STAGE2_DONE = False
state2 = read_state("ir")
if not RUN_STAGE_2:
    print("stage 2 skipped: RUN_STAGE_2 is False")
elif not STAGE1_DONE:
    print("stage 2 skipped: stage 1 is not finished")
elif state2.get("complete"):
    print("stage 2 is already complete")
else:
    budget = remaining_train_hours()
    est = epoch_hours("ir") or (epoch_hours("day") * IR_EPOCH_FACTOR if epoch_hours("day") else None)
    if est is None or est * EPOCH_SAFETY > budget:
        shown = f"~{est:.2f} h" if est is not None else "an unknown time"
        print(f"stage 2 not started: one epoch takes {shown} and {budget:.2f} h are available. Save Version and resume next session.")
    else:
        sh(f"python training/scripts/train.py --stage ir --data-root {DATA_ROOT} --run-dir {RUNS / 'ir'} --device {DEVICE} "
           f"--auto-resume --max-hours {budget:.3f} {WORKERS_FLAG}")
        sync_runs_to_drive()
    state2 = read_state("ir")
STAGE2_DONE = bool(state2.get("complete"))
if RUN_STAGE_2 and STAGE1_DONE:
    print(json.dumps({k: state2.get(k) for k in ("status", "epochs_done", "epochs_total", "early_stopped", "note")}, indent=2))
    if not STAGE2_DONE:
        print("Stage 2 is not finished. Save Version and resume in the next session.")
clock("after stage 2")

## 7. Evaluate on the validation split, day and IR separately; the hard-set regression gate

Runs only when the FINAL stage is complete (stage 2 when `RUN_STAGE_2`, else stage 1): a half-trained model is never evaluated,
exported or uploaded as "final". The test split is sealed until Phase 11 and is not touched. Validation also chose the checkpoint,
so these numbers are optimistic.

The hard-set gate compares against `hardset_baseline.json`. When no baseline exists it is recorded ONCE from the COCO-pretrained
`yolo11s.pt` (the model before fine-tuning, labelled `baseline=pretrained`), never from the model being judged.

In [ ]:
FINAL_STAGE = "ir" if RUN_STAGE_2 else "day"
READY = STAGE1_DONE and (STAGE2_DONE or not RUN_STAGE_2)
WEIGHTS = RUNS / FINAL_STAGE / "weights" / "best.pt"
GATE = None           # True passed, False failed, None not evaluated
if not READY:
    print(f"not evaluated: the final stage ({FINAL_STAGE}) is not complete yet")
elif not WEIGHTS.exists():
    raise RuntimeError(f"{WEIGHTS} is missing although the stage is marked complete")
else:
    clock("before evaluation")
    sh(f"python training/scripts/evaluate.py --weights {WEIGHTS} --tag final --device {DEVICE} --data-root {DATA_ROOT} {INDEX_FLAG}")
    if HARD_SET is None:
        print("HARD-SET GATE NOT EVALUATED: truewatch_ds/manifests/hard_set.txt was not found in the dataset input.")
    else:
        baseline = RESULTS / "hardset_baseline.json"
        source = RESULTS / "hardset_baseline_source.json"
        common = f"--hard-set {HARD_SET} --baseline {baseline} --device {DEVICE} --data-root {DATA_ROOT} {INDEX_FLAG}"
        if not baseline.exists():
            coco = CODE / "training" / "weights" / "yolo11s.pt"
            if not coco.exists():
                coco.parent.mkdir(parents=True, exist_ok=True)
                sh(f"python -c \"from ultralytics.utils.downloads import attempt_download_asset; attempt_download_asset('{coco}')\"")
            sha = lambda p: __import__("hashlib").sha256(Path(p).read_bytes()).hexdigest()
            if sha(coco) == sha(WEIGHTS):
                raise RuntimeError("the baseline weights are the weights being judged; refusing to record a self-baseline")
            sh(f"python training/scripts/eval_hardset.py --weights {coco} --tag baseline_pretrained --write-baseline {common}")
            source.write_text(json.dumps({
                "baseline": "pretrained",
                "weights": "yolo11s.pt, COCO-pretrained, before any TRUEWATCH fine-tuning",
                "weights_sha256": sha(coco),
                "why": "the gate needs a reference that is not the model being judged; Phase 2 must not regress below the pretrained model",
                "hard_set_sha256": HARD_SET_SHA,
                "recorded": time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime()),
            }, indent=2))
            print(f"hard-set baseline recorded from the COCO-pretrained model (baseline=pretrained): {baseline}")
        else:
            origin = json.loads(source.read_text()).get("baseline") if source.exists() else "UNKNOWN (no hardset_baseline_source.json)"
            print(f"hard-set baseline: {baseline} (baseline={origin})")
        rc = sh(f"python training/scripts/eval_hardset.py --weights {WEIGHTS} --tag final {common}", ok=(0, 1), check=False)
        if rc in (0, 1):
            report = json.loads((RESULTS / "hardset_final.json").read_text())
            GATE = (report.get("gate") or {}).get("passed")   # True / False; None only if no baseline was compared
        else:
            print(f"HARD-SET GATE REFUSED TO RUN (exit {rc}); see the messages above. Treated as failed.")
            GATE = False
        print(f"HARD-SET GATE: {'PASSED' if GATE else ('FAILED' if GATE is False else 'NOT EVALUATED')}")
    sh(f"python training/scripts/sweep_conf.py --preds {RESULTS / 'cache' / 'preds_final.npz'} --tag final --data-root {DATA_ROOT}")
    clock("after evaluation")

## 8. Export to ONNX (opset 17, dynamic batch) and check torch against onnxruntime

The parity line must read PASS (max absolute difference below 1e-3). The upload to the Hugging Face Hub happens only when the
`HF_TOKEN` Kaggle secret exists, a repo is named, and the hard-set gate did not fail.

In [ ]:
if READY:
    hf_token = secret("HF_TOKEN")
    hf_repo = HF_MODEL_REPO or secret("HF_MODEL_REPO")
    reasons = []
    if not hf_token:
        reasons.append("HF_TOKEN is not in Kaggle Secrets (Add-ons, Secrets)")
    if not hf_repo:
        reasons.append("no repo: set HF_MODEL_REPO above or add an HF_MODEL_REPO secret")
    if GATE is False and not UPLOAD_IF_GATE_FAILS:
        reasons.append("the hard-set regression gate failed")
    push = "" if reasons else f"--push-to-hub {hf_repo}"
    env = dict(os.environ, HF_TOKEN=hf_token) if hf_token else dict(os.environ)
    cmd = f"python training/scripts/export_onnx.py --weights {WEIGHTS} --tag final --parity-images {DATA_ROOT / 'images' / 'val'} {push}"
    print(f"$ {cmd}", flush=True)
    rc = subprocess.run(cmd, shell=True, cwd=str(CODE), env=env).returncode
    if rc != 0:
        raise RuntimeError(f"export failed with exit code {rc}")
    if reasons:
        print("NOT UPLOADED to the Hugging Face Hub: " + "; ".join(reasons) + ".\nUpload from the Mac instead: download best.pt from "
              "the results dataset, then HF_TOKEN=... python training/scripts/export_onnx.py --weights best.pt --tag final --push-to-hub <user>/<repo>")
else:
    print("export skipped: the final stage is not complete")
clock("after export")

## 9. Metrics report

Renders `training/results/METRICS.md`: measured against the slide-5 targets, with a verdict per target. Latency rows come from the
`benchmark_*.json` files committed in the repository, each labelled with the host it was measured on.

In [ ]:
if READY:
    benches = " ".join(f"--benchmark {p}" for p in sorted(glob.glob(str(RESULTS / "benchmark_*.json"))))
    hard = f"--hardset {RESULTS / 'hardset_final.json'}" if (RESULTS / "hardset_final.json").exists() else ""
    sh(
        f"python training/scripts/make_metrics.py --eval {RESULTS / 'eval_final.json'} {hard} "
        f"--sweep {RESULTS / 'sweep_final.json'} --export {RESULTS / 'export_final.json'} "
        f"--train-log {RUNS / FINAL_STAGE / 'train_log.csv'} {benches}"
    )
    print((RESULTS / "METRICS.md").read_text()[:6000])
else:
    print("metrics skipped: the final stage is not complete")

## 10. Package and publish the results dataset

Runs in every session, finished or not: checkpoints and logs are what the next session resumes from. Weights go to a private Kaggle
Dataset. Only the small metrics files and the model URL are meant to go back to git.

In [ ]:
if OUT.exists():
    shutil.rmtree(OUT)
(OUT / "runs").mkdir(parents=True)
for stage in ("day", "ir"):
    src = RUNS / stage
    if src.exists():
        shutil.copytree(src, OUT / "runs" / stage, ignore=shutil.ignore_patterns("*.jpg", "*.png", "epoch*.pt", "*.cache", "*.tmp"))
if RESULTS.exists():
    shutil.copytree(RESULTS, OUT / "results", ignore=shutil.ignore_patterns("cache"))
(OUT / "session.json").write_text(json.dumps({
    "git_ref": GIT_REF, "data_root": str(DATA_ROOT), "stage1_done": STAGE1_DONE, "stage2_done": STAGE2_DONE,
    "final_ready": READY, "hardset_gate": GATE, "hours_used": round(elapsed_h(), 2),
}, indent=2))

user, key = secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY")
if user and key:
    meta = {
        "title": "TRUEWATCH YOLO11-s runs",
        "id": f"{user}/truewatch-yolo11s-runs",
        "licenses": [{"name": "other"}],
        "subtitle": "Checkpoints, logs and metrics for the TRUEWATCH detector fine-tune",
    }
    (OUT / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
    os.environ.update(KAGGLE_USERNAME=user, KAGGLE_KEY=key)
    stamp = time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime())
    rc = sh(f"kaggle datasets version -p {OUT} -m 'session {stamp}' --dir-mode zip", check=False)
    if rc != 0:
        sh(f"kaggle datasets create -p {OUT} --dir-mode zip")
else:
    print("KAGGLE_USERNAME / KAGGLE_KEY not set. To keep the results: Save Version; this notebook's output (runs/ and outputs/) "
          "is what the next session attaches as an input.")
if ON_COLAB:
    sync_runs_to_drive()
    persist_out = PERSIST / "outputs"
    if persist_out.exists():
        shutil.rmtree(persist_out)
    shutil.copytree(OUT, persist_out)
    print(f"packaged outputs copied to {persist_out} (survives a Colab disconnect)")

print(sorted(str(p.relative_to(OUT)) for p in OUT.rglob("*") if p.is_file())[:60])
clock("end of session")
if elapsed_h() > SESSION_HOURS - 0.25:
    print("WARNING: this session ended within 15 min of the 12 h cap; lower TRAIN budgets or RESERVE_HOURS assumptions next time.")

## 11. What goes back to the repository

Commit only `training/results/*.json`, `*.csv`, `METRICS.md` and the Hugging Face model URL. Never a `.pt` or `.onnx`.
Download the small files from the results dataset, copy them into `training/results/` on your machine, and read `METRICS.md` before quoting any figure.